# Practical 7: ETL Pipelines: CSV and JSON Sources, Invalid Records, Validation and Incremental Loading

|  |  |
|---|---|
| **Name** | Aayushman Verma |
| **Roll No.** | 57 |
| **College** | B. K. Birla College, Kalyan |
| **Student ID** | 5176202 |

**Aim:** To build ETL pipelines for an HR payroll data set that extract data from one CSV file, from several CSV files and from a JSON document, clean and transform the data, find and remove invalid records, validate batches before loading, load them into a relational database and perform incremental loads using a watermark.

## Task 1: Create the Source Files and the Target Database

In [1]:
# Name: Ayushman Rajeev Verma
print("Name: Ayushman Rajeev Verma")
import os, glob, json, sqlite3
import numpy as np
import pandas as pd

HEAD = "emp_id,name,dept,monthly_salary,join_date,updated_at\n"
files = {
 "payroll_it.csv": [
    "E101,  aarav mehta ,IT,65000,2021-06-15,2026-01-05 10:00:00",
    "E102,ISHA NAIR,IT,72000,2019-03-01,2026-01-05 10:00:00",
    "E103,kabir singh,IT,-52000,2022-09-12,2026-01-06 09:30:00",         # negative salary
    "E101,  aarav mehta ,IT,65000,2021-06-15,2026-01-05 10:00:00"],      # duplicate row
 "payroll_hr.csv": [
    "E201,Meera Joshi,HR,48000,2020-11-23,2026-01-05 11:00:00",
    "E202,nikhil rao,HR,,2023-01-09,2026-01-05 11:00:00",                 # missing salary
    "E203,Pooja Desai,HR,51000,31-02-2022,2026-01-07 08:15:00"],         # invalid date
 "payroll_sales.csv": [
    "E301,Sameer Khan,Sales,41000,2022-07-18,2026-01-05 12:00:00",
    "E302,Tara Iyer,Sales,39000,2024-02-05,2026-01-08 14:20:00",
    "E303,Uday Patil,Sales,44500,2021-10-30,2026-01-08 14:45:00"],
}
for fname, lines in files.items():
    with open(fname, "w") as f:
        f.write(HEAD + "\n".join(lines) + "\n")

attendance = {"month": "2026-01", "working_days": 22, "records": [
    {"emp_id": "E101", "days_present": 21, "leaves": {"casual": 1, "sick": 0}},
    {"emp_id": "E102", "days_present": 22, "leaves": {"casual": 0, "sick": 0}},
    {"emp_id": "E201", "days_present": 18, "leaves": {"casual": 2, "sick": 2}},
    {"emp_id": "E301", "days_present": 20, "leaves": {"casual": 1, "sick": 1}},
    {"emp_id": "E999", "days_present": 22, "leaves": {"casual": 0, "sick": 0}},       # unknown employee
]}
json.dump(attendance, open("attendance.json", "w"), indent=2)

if os.path.exists("payroll_dw.db"):
    os.remove("payroll_dw.db")
db = sqlite3.connect("payroll_dw.db")
print("Created 3 department CSV files, attendance.json and the target database payroll_dw.db")

Name: Ayushman Rajeev Verma
Created 3 department CSV files, attendance.json and the target database payroll_dw.db


## Task 2: ETL for a Single CSV File

Three functions are reused in all later tasks: standardise() fixes text, dates and duplicates; enrich() adds annual salary, tax slab and tenure; to_db_types() prepares dates for SQLite.

In [2]:
# ---- reusable transformation functions
print("Name: Ayushman Rajeev Verma")
def standardise(df):
    out = df.copy()
    out["name"] = out["name"].str.strip().str.title()
    out["dept"] = out["dept"].str.strip().str.upper()
    out["join_date"] = pd.to_datetime(out["join_date"], format="%Y-%m-%d", errors="coerce")
    out["updated_at"] = pd.to_datetime(out["updated_at"])
    return out.drop_duplicates()

def enrich(df):
    out = df.copy()
    out["annual_salary"] = out["monthly_salary"] * 12
    out["tax_pct"] = np.select([out["monthly_salary"] <= 40000, out["monthly_salary"] <= 70000], [5, 10], default=20)
    out["tenure_years"] = ((pd.Timestamp("2026-04-01") - out["join_date"]).dt.days / 365.25).round(1)
    return out

def to_db_types(df):                       # SQLite stores dates as text
    out = df.copy()
    out["join_date"] = out["join_date"].dt.strftime("%Y-%m-%d")
    out["updated_at"] = out["updated_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
    return out

# ---- E X T R A C T
raw_it = pd.read_csv("payroll_it.csv")
print("Extracted:", len(raw_it), "rows\n", raw_it.to_string(index=False), "\n")

# ---- T R A N S F O R M
it = enrich(standardise(raw_it))
print("Transformed:", len(it), "rows (duplicate removed)\n", it[["emp_id", "name", "dept", "monthly_salary", "tax_pct", "tenure_years"]].to_string(index=False), "\n")

# ---- L O A D
to_db_types(it).to_sql("payroll_it", db, if_exists="replace", index=False)
print("Loaded into table payroll_it ->", db.execute("SELECT COUNT(*) FROM payroll_it").fetchone()[0], "rows")

Name: Ayushman Rajeev Verma
Extracted: 4 rows
 emp_id           name dept  monthly_salary  join_date          updated_at
  E101   aarav mehta    IT           65000 2021-06-15 2026-01-05 10:00:00
  E102      ISHA NAIR   IT           72000 2019-03-01 2026-01-05 10:00:00
  E103    kabir singh   IT          -52000 2022-09-12 2026-01-06 09:30:00
  E101   aarav mehta    IT           65000 2021-06-15 2026-01-05 10:00:00 

Transformed: 3 rows (duplicate removed)
 emp_id        name dept  monthly_salary  tax_pct  tenure_years
  E101 Aarav Mehta   IT           65000       10           4.8
  E102   Isha Nair   IT           72000       20           7.1
  E103 Kabir Singh   IT          -52000        5           3.6 

Loaded into table payroll_it -> 3 rows


## Task 3: Extract Multiple CSV Files and Combine Them

In [3]:
print("Name: Ayushman Rajeev Verma")
parts = {}
for path in sorted(glob.glob("payroll_*.csv")):
    parts[os.path.basename(path)] = pd.read_csv(path)
    print(f"{path:<18} rows: {len(parts[path]):>2}")

combined = pd.concat(parts, names=["source_file", "row"]).reset_index(level="source_file")    # keeps the file name
print("\nRows combined:", len(combined))
combined = enrich(standardise(combined))
print("Rows after standardise + de-duplicate:", len(combined))

to_db_types(combined).to_sql("payroll_all", db, if_exists="replace", index=False)
print("\nRows per department in table payroll_all:")
print(pd.read_sql("SELECT dept, COUNT(*) AS employees, ROUND(AVG(monthly_salary)) AS avg_salary FROM payroll_all GROUP BY dept", db).to_string(index=False))

Name: Ayushman Rajeev Verma
payroll_hr.csv     rows:  3
payroll_it.csv     rows:  4
payroll_sales.csv  rows:  3

Rows combined: 10
Rows after standardise + de-duplicate: 9

Rows per department in table payroll_all:
 dept  employees  avg_salary
   HR          3     49500.0
   IT          3     28333.0
SALES          3     41500.0


## Task 4: Extract and Flatten a JSON Document

In [4]:
print("Name: Ayushman Rajeev Verma")
doc = json.load(open("attendance.json"))
att = pd.json_normalize(doc, record_path="records", meta=["month", "working_days"], sep="_")   # flatten nested leaves
att["attendance_pct"] = (att["days_present"] / att["working_days"] * 100).round(1)
att["total_leaves"] = att["leaves_casual"] + att["leaves_sick"]
att = att[["emp_id", "month", "days_present", "total_leaves", "attendance_pct"]]
print(att.to_string(index=False))

att.to_sql("attendance", db, if_exists="replace", index=False)
print("\nJoin check - attendance of known employees only:")
print(pd.read_sql("""SELECT a.emp_id, p.name, p.dept, a.attendance_pct FROM attendance a
                     JOIN payroll_all p ON p.emp_id = a.emp_id ORDER BY a.emp_id""", db).to_string(index=False))

Name: Ayushman Rajeev Verma
emp_id   month  days_present  total_leaves attendance_pct
  E101 2026-01            21             1      95.454545
  E102 2026-01            22             0          100.0
  E201 2026-01            18             4      81.818182
  E301 2026-01            20             2      90.909091
  E999 2026-01            22             0          100.0

Join check - attendance of known employees only:
emp_id        name  dept  attendance_pct
  E101 Aarav Mehta    IT       95.454545
  E102   Isha Nair    IT      100.000000
  E201 Meera Joshi    HR       81.818182
  E301 Sameer Khan SALES       90.909091


## Task 5: Identify and Remove Invalid Records (Rule Based)

Every rule is a message plus a function. A row that breaks a rule goes to the payroll_rejects table together with the reasons.

In [5]:
# Each rule is (message, function that is True for a valid row)
print("Name: Ayushman Rajeev Verma")
RULES = {
    "monthly_salary must be a positive number": lambda r: pd.notna(r["monthly_salary"]) and r["monthly_salary"] > 0,
    "join_date must be a valid date":           lambda r: pd.notna(r["join_date"]),
    "name must not be empty":                   lambda r: bool(str(r["name"]).strip()),
}

def split_valid_invalid(df):
    reasons = df.apply(lambda r: "; ".join(msg for msg, ok in RULES.items() if not ok(r)), axis=1)
    return df[reasons == ""], df[reasons != ""].assign(reason=reasons[reasons != ""])

valid, invalid = split_valid_invalid(combined)
print(f"Total: {len(combined)} | valid: {len(valid)} | invalid: {len(invalid)}\n")
print("Rejected records:")
print(invalid[["emp_id", "name", "reason"]].to_string(index=False))
to_db_types(invalid).to_sql("payroll_rejects", db, if_exists="replace", index=False)

Name: Ayushman Rajeev Verma
Total: 9 | valid: 6 | invalid: 3

Rejected records:
emp_id        name                                   reason
  E202  Nikhil Rao monthly_salary must be a positive number
  E203 Pooja Desai           join_date must be a valid date
  E103 Kabir Singh monthly_salary must be a positive number


3

## Task 6: Validate Data Before Loading to the Target Database

validate_batch() raises a ValidationError when a batch is unsafe to load; the load function catches it, reports the reason and loads nothing. The target table also has a primary key and a CHECK constraint.

In [6]:
print("Name: Ayushman Rajeev Verma")
class ValidationError(Exception):
    pass

REQUIRED = ["emp_id", "name", "dept", "monthly_salary", "join_date", "updated_at"]

def validate_batch(df):
    """Raises ValidationError when the batch must NOT be loaded."""
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValidationError(f"missing columns {missing}")
    problems = []
    if df["emp_id"].duplicated().any():          problems.append("duplicate emp_id")
    if df["monthly_salary"].isna().any() or (df["monthly_salary"] <= 0).any():
        problems.append("invalid salary")
    if problems:
        raise ValidationError("; ".join(problems))

db.executescript("""
DROP TABLE IF EXISTS payroll_clean;
CREATE TABLE payroll_clean (emp_id TEXT PRIMARY KEY, name TEXT NOT NULL, dept TEXT, monthly_salary REAL CHECK (monthly_salary > 0),
    join_date TEXT, updated_at TEXT, annual_salary REAL, tax_pct INTEGER, tenure_years REAL);
""")
COLS = ["emp_id", "name", "dept", "monthly_salary", "join_date", "updated_at", "annual_salary", "tax_pct", "tenure_years"]

def guarded_load(batch, label):
    try:
        validate_batch(batch)
    except ValidationError as err:
        print(f"[{label}] REJECTED  -> {err}")
        return
    to_db_types(batch)[COLS].to_sql("payroll_clean", db, if_exists="append", index=False)
    print(f"[{label}] ACCEPTED  -> {len(batch)} rows loaded")

guarded_load(valid.drop(columns=["monthly_salary"]), "batch 1: salary column missing")
guarded_load(pd.concat([valid, valid.head(1)]), "batch 2: repeated employee")
guarded_load(invalid.drop(columns="reason"), "batch 3: bad salaries")
guarded_load(valid, "batch 4: clean data")
print("\nTarget table payroll_clean:")
print(pd.read_sql("SELECT emp_id, name, dept, monthly_salary, tax_pct, updated_at FROM payroll_clean", db).to_string(index=False))

Name: Ayushman Rajeev Verma
[batch 1: salary column missing] REJECTED  -> missing columns ['monthly_salary']
[batch 2: repeated employee] REJECTED  -> duplicate emp_id
[batch 3: bad salaries] REJECTED  -> invalid salary
[batch 4: clean data] ACCEPTED  -> 6 rows loaded

Target table payroll_clean:
emp_id        name  dept  monthly_salary  tax_pct          updated_at
  E201 Meera Joshi    HR         48000.0       10 2026-01-05 11:00:00
  E101 Aarav Mehta    IT         65000.0       10 2026-01-05 10:00:00
  E102   Isha Nair    IT         72000.0       20 2026-01-05 10:00:00
  E301 Sameer Khan SALES         41000.0       10 2026-01-05 12:00:00
  E302   Tara Iyer SALES         39000.0        5 2026-01-08 14:20:00
  E303  Uday Patil SALES         44500.0       10 2026-01-08 14:45:00


## Task 7: Incremental Loading with a Watermark

The watermark table stores the newest updated_at that was loaded. A new run only processes rows that are newer, updates changed employees and inserts new ones.

In [7]:
# Incremental load using a WATERMARK (the newest updated_at that was already loaded)
print("Name: Ayushman Rajeev Verma")
db.execute("CREATE TABLE IF NOT EXISTS etl_watermark (table_name TEXT PRIMARY KEY, last_loaded TEXT)")
db.execute("INSERT OR REPLACE INTO etl_watermark SELECT 'payroll_clean', MAX(updated_at) FROM payroll_clean")
db.commit()

with open("payroll_update.csv", "w") as f:
    f.write(HEAD + "\n".join([
        "E101,Aarav Mehta,IT,68000,2021-06-15,2026-01-10 09:00:00",     # salary revised -> changed
        "E302,Tara Iyer,Sales,39000,2024-02-05,2026-01-08 14:20:00",    # unchanged (older than watermark)
        "E304,Vikram Sethi,Sales,46000,2025-12-01,2026-01-11 10:00:00", # new employee
        "E201,Meera Joshi,HR,50000,2020-11-23,2026-01-12 16:30:00"]) + "\n")  # salary revised

def incremental_load(path):
    mark = db.execute("SELECT last_loaded FROM etl_watermark WHERE table_name = 'payroll_clean'").fetchone()[0]
    incoming = enrich(standardise(pd.read_csv(path)))
    fresh = incoming[incoming["updated_at"] > pd.Timestamp(mark)]            # only rows newer than the watermark
    print(f"  watermark = {mark} | incoming = {len(incoming)} | newer = {len(fresh)} | skipped = {len(incoming) - len(fresh)}")
    if fresh.empty:
        return
    validate_batch(fresh)
    db.executemany("""INSERT INTO payroll_clean VALUES (?,?,?,?,?,?,?,?,?) ON CONFLICT(emp_id) DO UPDATE SET
                      monthly_salary = excluded.monthly_salary, updated_at = excluded.updated_at,
                      annual_salary = excluded.annual_salary, tax_pct = excluded.tax_pct""",
                   to_db_types(fresh)[COLS].astype(object).values.tolist())
    db.execute("UPDATE etl_watermark SET last_loaded = (SELECT MAX(updated_at) FROM payroll_clean) WHERE table_name = 'payroll_clean'")
    db.commit()

print("Rows before:", db.execute("SELECT COUNT(*) FROM payroll_clean").fetchone()[0])
print("Run 1 - new file arrives:")
incremental_load("payroll_update.csv")
print("Run 2 - same file again:")
incremental_load("payroll_update.csv")
print("\nRows after:", db.execute("SELECT COUNT(*) FROM payroll_clean").fetchone()[0])
print(pd.read_sql("SELECT emp_id, name, monthly_salary, tax_pct, updated_at FROM payroll_clean ORDER BY emp_id", db).to_string(index=False))

Name: Ayushman Rajeev Verma
Rows before: 6
Run 1 - new file arrives:
  watermark = 2026-01-08 14:45:00 | incoming = 4 | newer = 3 | skipped = 1
Run 2 - same file again:
  watermark = 2026-01-12 16:30:00 | incoming = 4 | newer = 0 | skipped = 4

Rows after: 7
emp_id         name  monthly_salary  tax_pct          updated_at
  E101  Aarav Mehta         68000.0       10 2026-01-10 09:00:00
  E102    Isha Nair         72000.0       20 2026-01-05 10:00:00
  E201  Meera Joshi         50000.0       10 2026-01-12 16:30:00
  E301  Sameer Khan         41000.0       10 2026-01-05 12:00:00
  E302    Tara Iyer         39000.0        5 2026-01-08 14:20:00
  E303   Uday Patil         44500.0       10 2026-01-08 14:45:00
  E304 Vikram Sethi         46000.0       10 2026-01-11 10:00:00


**Conclusion:** ETL pipelines were built for CSV and JSON sources. The data was standardised and enriched, files were combined, invalid records were rule-checked and stored with reasons, unsafe batches were blocked by a validation step before loading, and the watermark technique loaded only new or changed employees on each run instead of reloading everything.